# 🛡️ RAG-Based Sigma Rule Generator for Cyber Threat Intelligence

**Pipeline Overview:**
1. Load the `reloading0101/threat-intelligence-dataset` from HuggingFace (~52k CTI examples)
2. Embed documents using `cisco-ai/SecureBERT2.0-base` (domain-specific cybersecurity BERT)
3. Store and index embeddings in a persistent **ChromaDB** vector store
4. Given a CTI query/report, retrieve the most semantically relevant context
5. Build a structured prompt and send to **sylink:32b** running locally via **Ollama**
6. Output a valid **Sigma detection rule** in YAML format

## 1. Install Dependencies

In [2]:
!pip install --upgrade pip
!pip install datasets pandas tqdm numpy
!pip install torch torchvision
!pip install transformers accelerate tokenizers
!pip install chromadb
!pip install requests

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 11.3 MB/s  0:00:00


ERROR: To modify pip, please run the following command:
C:\Users\Lawliett\miniconda3\envs\cybersecrag\python.exe -m pip install --upgrade pip


## 2. Load the Threat Intelligence Dataset

Dataset: [`reloading0101/threat-intelligence-dataset`](https://huggingface.co/datasets/reloading0101/threat-intelligence-dataset)

~52,279 instruction-response pairs across 38 cybersecurity categories.

Each document is built as: `instruction + input + output` for maximum context richness.

In [3]:
from datasets import load_dataset
import pandas as pd

print("Loading dataset from HuggingFace...")
dataset = load_dataset("reloading0101/threat-intelligence-dataset", split="train")
print(f"Total records loaded: {len(dataset)}")

df = pd.DataFrame(dataset)
print(f"Columns: {df.columns.tolist()}")
df.head(2)

Loading dataset from HuggingFace...
Total records loaded: 52279
Columns: ['instruction', 'input', 'output', 'metadata']


,instruction,input,output,metadata
0,Analyze this cyber campaign and provide intell...,Campaign: Operation Dragon Night\nThreat Actor...,Campaign Analysis: Operation Dragon Night\n\nE...,"{'category': 'campaign-analysis', 'source': 'a..."
1,What are the key indicators and TTPs for this ...,Operation Ghost Strike,Threat Intelligence Report: Operation Ghost St...,"{'category': 'campaign-analysis', 'source': 'a..."


In [4]:
# Explore metadata categories
if "metadata" in df.columns:
    categories = df["metadata"].apply(lambda x: x.get("category") if isinstance(x, dict) else None)
    print("Category distribution:")
    print(categories.value_counts().to_string())
else:
    print("No metadata column found. Available columns:", df.columns.tolist())

Category distribution:
metadata
campaign-analysis                 4867
insider-threats                   2421
threat-modeling                   1924
threat-hunting                    1857
botnet-infrastructure             1841
ransomware-operations             1817
zero-day-exploits                 1808
attribution-analysis              1792
api-security                      1769
container-security                1736
email-threats                     1704
web-application-security          1698
geopolitical-threats              1660
cryptojacking-mining              1658
deception-technology              1512
data-exfiltration                 1442
cloud-saas-security               1389
digital-risk-management           1379
red-team-operations               1256
ai-ml-threats                     1235
threat-intelligence-feeds         1216
social-engineering-fraud          1211
network-based-threats             1182
supply-chain-attacks              1132
threat-actors                   

In [5]:
# Build full-context documents: [category] + instruction + input + output
# Category tag prefix helps SecureBERT2.0 anchor the semantic space correctly.
# Output field is placed last (highest weight in mean pooling) since it
# contains the densest technical content: IOCs, TTPs, CVEs, MITRE mappings.

def build_document(row):
    """
    Construct a rich embedding document from a dataset row.

    Format:
        [category-tag]\n
        Instruction: ...\n
        Input: ...\n          (omitted if empty)
        Response: ...

    The category tag prefix helps the model anchor to the correct
    cybersecurity subdomain during retrieval.
    """
    parts = []

    # Extract category from metadata if available
    category = None
    if isinstance(row.get("metadata"), dict):
        category = row["metadata"].get("category")
    if category:
        parts.append(f"[{category}]")

    if pd.notna(row.get("instruction")) and str(row["instruction"]).strip():
        parts.append(f"Instruction: {str(row['instruction']).strip()}")

    if pd.notna(row.get("input")) and str(row["input"]).strip():
        parts.append(f"Input: {str(row['input']).strip()}")

    if pd.notna(row.get("output")) and str(row["output"]).strip():
        parts.append(f"Response: {str(row['output']).strip()}")

    return "\n".join(parts)


df["document"] = df.apply(build_document, axis=1)
df.dropna(subset=["document"], inplace=True)
df = df[df["document"].str.strip() != ""]
df.reset_index(drop=True, inplace=True)

documents = df["document"].tolist()
ids = [str(i) for i in range(len(documents))]
nb = len(documents)

print(f"Total documents prepared for embedding: {nb}")
print("\n--- Sample document (first 800 chars) ---")
print(documents[0][:800], "...")

Total documents prepared for embedding: 52279

--- Sample document (first 800 chars) ---
[campaign-analysis]
Instruction: Analyze this cyber campaign and provide intelligence summary.
Input: Campaign: Operation Dragon Night
Threat Actor: Pioneer Kitten
Response: Campaign Analysis: Operation Dragon Night

Executive Summary:
Operation Dragon Night is an ongoing cyber espionage campaign attributed to Pioneer Kitten. The campaign has been active since 2025-07-28 and primarily targets Energy and Manufacturing sectors across North America.

Attack Chain:
1. Initial Access: Compromised websites (watering hole)
2. Execution: CobaltStrike dropper executes via DLL side-loading
3. Persistence: WMI event subscription
4. Credential Access: Keylogging
5. Lateral Movement: Pass-the-hash
6. Exfiltration: Data staged and transferred to 78.225.160.232

Malware Used:
- Primary: CobaltStrike
- Se ...


## 3. SecureBERT2.0 Embedder

Model: [`cisco-ai/SecureBERT2.0-base`](https://huggingface.co/cisco-ai/SecureBERT2.0-base)

A BERT-based model pre-trained on large-scale cybersecurity corpora by Cisco AI Research.
Produces 768-dimensional embeddings via **mean pooling** over token representations.

**Batching strategy:** Texts are processed in batches of 16 to avoid OOM errors.
Documents are truncated to 512 tokens (BERT's max context window).

In [6]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

class SecureBERT2Embedder:
    """
    Embedding wrapper for cisco-ai/SecureBERT2.0-base.
    Uses mean pooling over the last hidden state, weighted by the attention mask,
    to produce a single fixed-size vector per document.
    """

    def __init__(self, model_name: str = "cisco-ai/SecureBERT2.0-base", batch_size: int = 16):
        print(f"Loading tokenizer and model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()
        self.batch_size = batch_size

        # Use GPU if available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        print(f"Model loaded. Using device: {self.device}")

    def mean_pooling(self, model_output, attention_mask):
        """Weighted mean of token embeddings using attention mask."""
        token_embeddings = model_output.last_hidden_state  # (batch, seq_len, hidden_size)
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask  # (batch, hidden_size)

    def encode(self, texts: list) -> np.ndarray:
        """Encode a list of strings into embedding vectors."""
        all_embeddings = []
        with torch.no_grad():
            for i in range(0, len(texts), self.batch_size):
                batch = texts[i : i + self.batch_size]
                encoded_input = self.tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt",
                ).to(self.device)
                model_output = self.model(**encoded_input)
                embeddings = self.mean_pooling(model_output, encoded_input["attention_mask"])
                all_embeddings.extend(embeddings.cpu().numpy())
        return np.array(all_embeddings)


In [7]:
# Initialize the embedder (downloads model weights on first run ~440MB)
embedder = SecureBERT2Embedder()

Loading tokenizer and model: cisco-ai/SecureBERT2.0-base


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: cisco-ai/SecureBERT2.0-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Using device: cpu


In [8]:
# Quick sanity check — embed a single test sentence
test_embedding = embedder.encode(["Test: SQL injection attack via HTTP POST parameter."])
print(f"Embedding shape: {test_embedding.shape}")   # Expected: (1, 768)
print(f"Embedding dtype: {test_embedding.dtype}")
print(f"Sample values: {test_embedding[0][:5]}")

Embedding shape: (1, 768)
Embedding dtype: float32
Sample values: [ 0.25881854 -0.97059655 -0.24775068  0.25316432  0.46763396]


## 4. ChromaDB Vector Store Setup

Using a persistent local ChromaDB client.
Collection: `cti_db_v2`

Documents are only added if the collection doesn't already exist, making this cell **safe to re-run**.

In [9]:
import chromadb

DB_PATH = "db_v2/"
COLLECTION_NAME = "cti_db_v2"

client = chromadb.PersistentClient(path=DB_PATH)
print(f"Existing collections: {client.list_collections()}")

Existing collections: [Collection(name=cti_db_v2)]


In [10]:
collection_exists = False

try:
    collection = client.get_collection(name=COLLECTION_NAME)
    print(f"✅ Collection '{COLLECTION_NAME}' already exists with {collection.count()} documents.")
    collection_exists = True
except Exception:
    print(f"Collection '{COLLECTION_NAME}' not found. Creating...")
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}  # Cosine similarity for semantic search
    )
    print(f"✅ Collection '{COLLECTION_NAME}' created.")

✅ Collection 'cti_db_v2' already exists with 52279 documents.


## 5. Embed & Index Documents

This step embeds all ~52k documents and inserts them into ChromaDB in batches.

⚠️ **This will take time** depending on your hardware:
- CPU: ~30–60 minutes
- GPU (CUDA): ~5–10 minutes

It is **skipped automatically** if the collection already exists.

In [11]:
from tqdm import tqdm

CHROMA_BATCH_SIZE = 5000   # How many docs to upsert into ChromaDB per call
EMBED_BATCH_SIZE  = 16     # How many docs to embed per forward pass

if not collection_exists:
    print(f"Embedding and indexing {nb} documents...")
    embedder.batch_size = EMBED_BATCH_SIZE

    for start in tqdm(range(0, nb, CHROMA_BATCH_SIZE), desc="Indexing batches"):
        end = min(start + CHROMA_BATCH_SIZE, nb)

        batch_ids   = ids[start:end]
        batch_docs  = documents[start:end]

        # Embed the current batch
        batch_embeddings = embedder.encode(batch_docs)

        # Insert into ChromaDB
        collection.add(
            ids=batch_ids,
            documents=batch_docs,
            embeddings=batch_embeddings.tolist(),
        )

    print(f"✅ Done! Total documents in collection: {collection.count()}")
else:
    print("ℹ️  Collection already populated — skipping embedding step.")

ℹ️  Collection already populated — skipping embedding step.


## 6. RAG Retrieval

Given a CTI report or threat description:
1. Embed the query using SecureBERT2.0
2. Query ChromaDB for the top-K most semantically similar documents
3. Return the retrieved context to augment the LLM prompt

In [12]:
def retrieve_context(query_text: str, top_k: int = 5) -> str:
    """
    Embed a query string and retrieve the top-K most similar documents
    from ChromaDB. Returns a single concatenated context string.

    Parameters
    ----------
    query_text : str
        The CTI report, threat description, or user question.
    top_k : int
        Number of documents to retrieve (default: 5).

    Returns
    -------
    str
        Retrieved documents joined by separator lines.
    """
    query_embedding = embedder.encode([query_text])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
    )

    docs = results["documents"][0]
    context = "\n\n---\n\n".join(docs)
    return context


print("retrieve_context() ready.")

retrieve_context() ready.


## 7. Prompt Engineering for Sigma Rule Generation

The prompt instructs the LLM to act as a cybersecurity expert and:
- Analyze both the user query and the RAG-retrieved context
- Extract relevant IOCs, TTPs, and MITRE ATT&CK mappings
- Output **only** a valid Sigma rule in YAML format

In [13]:
from enum import Enum

class OutputType(str, Enum):
    SIGMA      = "sigma"
    YARA       = "yara"
    CTI_REPORT = "cti_report"


# ── Per-output-type instructions ──────────────────────────────────────────────

_SIGMA_INSTRUCTIONS = """\
Your task:
- Carefully analyze BOTH the query and the retrieved context
- Extract only the relevant IOCs, TTPs, and MITRE ATT&CK patterns
- Map extracted data to Sigma rule detection components:
  - `logsource` → correct product, category, or service type
  - `detection` → precise match conditions using Sigma syntax
  - Include filters to reduce false positives where applicable

Strict constraints:
- Output ONLY valid YAML — no markdown fences, no explanations, no extra text
- Use these fields in this exact order: title, id, status, description, references, tags, author, date, logsource, detection, fields, falsepositives, level
- id must be a valid UUID v4 (generate one)
- level must be one of: informational, low, medium, high, critical
- status must be one of: stable, test, experimental
- Use exact field names from the CTI input — do not invent values
- Where multiple IOCs exist, include them in YAML lists
- Use correct Sigma key conventions (e.g., http_method, url_path, CommandLine, Image)

Output format:
title: <short capitalised title under 60 characters>
id: <uuid-v4>
status: experimental
description: <one-sentence description of what this rule detects>
references:
    - <CVE URL, MITRE URL, or vendor advisory>
tags:
    - attack.<tactic>
    - attack.<technique_id>
author: <author name>
date: <YYYY/MM/DD>
logsource:
    category: <e.g., webserver, process_creation, network_connection>
    product: <e.g., windows, linux, cisco>
detection:
    selection:
        FieldName|contains:
            - 'value1'
            - 'value2'
    condition: selection
fields:
    - <log fields relevant for analyst investigation>
falsepositives:
    - <legitimate scenario that could trigger this rule>
level: <informational|low|medium|high|critical>

Output only the Sigma rule YAML below:"""


_YARA_INSTRUCTIONS = """\
Your task:
- Carefully analyze BOTH the query and the retrieved context
- Extract file-level, memory, or network IOCs suitable for YARA matching:
  strings (hex patterns, ASCII, wide), byte sequences, file metadata, PE sections
- Build a precise YARA rule that minimises false positives

Strict constraints:
- Output ONLY valid YARA syntax — no markdown fences, no explanations
- Include a `meta` section with: description, author, date, reference, hash (if available)
- Include a `strings` section with relevant patterns (use `nocase`, `wide`, `ascii` modifiers where appropriate)
- Include a `condition` section — prefer `any of them` or specific count thresholds
- Rule name must be snake_case, descriptive, under 60 characters
- Comment each string with what it represents

Output format:
rule <rule_name>
{
    meta:
        description = "<what this rule detects>"
        author      = "<author>"
        date        = "<YYYY-MM-DD>"
        reference   = "<CVE, MITRE, or source URL>"
        hash        = "<sample hash if available, else N/A>"

    strings:
        $s1 = "<string_value>" ascii nocase  // <what it represents>
        $h1 = { <hex bytes> }                 // <what it represents>

    condition:
        <condition expression>
}

Output only the YARA rule below:"""


_CTI_REPORT_INSTRUCTIONS = """\
Your task:
- Carefully analyze BOTH the query and the retrieved context
- Produce a structured, professional Cyber Threat Intelligence (CTI) report
  with actionable mitigation steps

Report must include these sections in order:
1. **Executive Summary** — 2-3 sentence overview of the threat
2. **Threat Actor / Campaign** — known attribution or TTPs if available
3. **Technical Analysis** — exploitation mechanism, affected systems, attack chain
4. **MITRE ATT&CK Mapping** — tactic → technique → sub-technique table
5. **Indicators of Compromise (IOCs)** — IPs, domains, hashes, URLs, registry keys
6. **Mitigation & Hardening Steps** — prioritized, actionable recommendations
7. **Detection Opportunities** — log sources and behavioral signals to monitor
8. **References** — CVEs, vendor advisories, threat reports

Constraints:
- Use markdown formatting (headers, tables, bullet lists)
- Be concise but technically precise — target audience is a SOC analyst
- Only include information supported by the query or retrieved context
- Do not invent IOCs or attribution

Output the CTI report below:"""


_INSTRUCTIONS = {
    OutputType.SIGMA:      _SIGMA_INSTRUCTIONS,
    OutputType.YARA:       _YARA_INSTRUCTIONS,
    OutputType.CTI_REPORT: _CTI_REPORT_INSTRUCTIONS,
}

_ROLE = {
    OutputType.SIGMA:      "You are a cybersecurity expert specializing in Sigma rule creation and threat detection engineering.",
    OutputType.YARA:       "You are a malware analyst and threat detection engineer specializing in YARA rule development.",
    OutputType.CTI_REPORT: "You are a senior Cyber Threat Intelligence analyst producing actionable reports for SOC and incident response teams.",
}


def build_prompt(original_query: str, context: str, output_type: OutputType = OutputType.SIGMA) -> str:
    """
    Construct the full LLM prompt for the selected output type.

    Parameters
    ----------
    original_query : str
        The CTI report, threat description, or incident notes.
    context : str
        Retrieved threat intelligence context from ChromaDB.
    output_type : OutputType
        One of OutputType.SIGMA, OutputType.YARA, OutputType.CTI_REPORT.

    Returns
    -------
    str
        The fully assembled prompt string.
    """
    role         = _ROLE[output_type]
    instructions = _INSTRUCTIONS[output_type]

    return f"""{role}
You will receive:
1. A CTI report or threat description (user query)
2. Relevant threat intelligence context retrieved from a RAG knowledge base

{instructions}

USER QUERY (CTI Report):
\"\"\"
{original_query}
\"\"\"

RETRIEVED THREAT INTELLIGENCE CONTEXT:
\"\"\"
{context}
\"\"\"
"""


# ── Quick test ────────────────────────────────────────────────────────────────
print("OutputType options:")
for ot in OutputType:
    print(f"  OutputType.{ot.name} → '{ot.value}'")
print("\nbuild_prompt() ready.")

OutputType options:
  OutputType.SIGMA → 'sigma'
  OutputType.YARA → 'yara'
  OutputType.CTI_REPORT → 'cti_report'

build_prompt() ready.


## 8. Generate Sigma Rule via Ollama (sylink:8b)

Sends the full prompt to **sylink:8b** running locally through Ollama's REST API.

**Prerequisites:**
- Ollama must be running: `ollama serve`
- Model must be pulled: `ollama pull sylink:32b`
- Default endpoint: `http://localhost:11434/api/generate`

Temperature is set low (0.1) for deterministic, structured YAML output.

In [14]:
import requests
import json

OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "sylink/sylink:8b"   # ← run `ollama list` and paste the exact tag here

def generate_sigma_rule(
    prompt: str,
    model: str = OLLAMA_MODEL,
    temperature: float = 0.1,
    timeout: int = 300,
) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "temperature": temperature,
        "stream": False,
        "options": {
            "num_predict": 2048,
        },
    }

    print(f"Sending request to Ollama ({model})...")
    response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    response.raise_for_status()

    data = response.json()
    return data.get("response", "").strip()


print("generate_sigma_rule() ready.")

generate_sigma_rule() ready.


## 9. Validate & Save the Sigma Rule

Basic YAML validation and save to a `.yml` file.

In [15]:
import yaml
import re
import os
from datetime import datetime


def clean_output(raw_output: str) -> str:
    """Strip markdown code fences if the model wrapped the output in them."""
    cleaned = re.sub(r"^```[a-zA-Z]*\n?", "", raw_output.strip())
    cleaned = re.sub(r"\n?```$", "", cleaned.strip())
    return cleaned.strip()


def validate_output(output_str: str, output_type: OutputType) -> bool:
    """
    Validate the LLM output according to its type.
    Returns True if valid, False otherwise.
    """
    if output_type == OutputType.SIGMA:
        required_fields = {"title", "id", "description", "logsource", "detection", "level"}
        try:
            rule = yaml.safe_load(output_str)
            if not isinstance(rule, dict):
                print("❌ Sigma: Parsed YAML is not a dictionary.")
                return False
            missing = required_fields - set(rule.keys())
            if missing:
                print(f"⚠️  Sigma: Missing fields: {missing}")
            else:
                print("✅ Sigma: All required fields present.")
            return True
        except yaml.YAMLError as e:
            print(f"❌ Sigma: YAML parsing error: {e}")
            return False

    elif output_type == OutputType.YARA:
        has_rule    = "rule " in output_str
        has_strings = "strings:" in output_str
        has_cond    = "condition:" in output_str
        has_meta    = "meta:" in output_str
        checks = {"rule keyword": has_rule, "strings section": has_strings,
                  "condition section": has_cond, "meta section": has_meta}
        all_ok = all(checks.values())
        for name, ok in checks.items():
            print(f"{'✅' if ok else '❌'} YARA: {name}")
        return all_ok

    elif output_type == OutputType.CTI_REPORT:
        has_exec    = "Executive Summary" in output_str
        has_mitre   = "MITRE" in output_str or "ATT&CK" in output_str
        has_iocs    = "IOC" in output_str or "Indicator" in output_str
        has_mitig   = "Mitigation" in output_str or "Hardening" in output_str
        checks = {"Executive Summary section": has_exec, "MITRE mapping": has_mitre,
                  "IOC section": has_iocs, "Mitigation section": has_mitig}
        all_ok = all(checks.values())
        for name, ok in checks.items():
            print(f"{'✅' if ok else '⚠️ '} CTI Report: {name}")
        return all_ok

    return True


_EXTENSIONS = {
    OutputType.SIGMA:      ".yml",
    OutputType.YARA:       ".yar",
    OutputType.CTI_REPORT: ".md",
}

_OUTPUT_DIRS = {
    OutputType.SIGMA:      "sigma_rules/",
    OutputType.YARA:       "yara_rules/",
    OutputType.CTI_REPORT: "cti_reports/",
}


def save_output(output_str: str, output_type: OutputType) -> str:
    """Save the LLM output to an appropriately named file."""
    out_dir = _OUTPUT_DIRS[output_type]
    ext     = _EXTENSIONS[output_type]
    os.makedirs(out_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix    = output_type.value
    filepath  = os.path.join(out_dir, f"{prefix}_{timestamp}{ext}")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(output_str)
    print(f"✅ Saved to: {filepath}")
    return filepath


print("Validation and save utilities ready.")

Validation and save utilities ready.


## 10. Interactive Query Interface

Reusable end-to-end function: provide any CTI text, get a Sigma rule back.

In [17]:
example_cti = """
Distribution of HijackLoader with IObit’s
Driver Booster executable
In May and June 2024, security researchers from Lab52 and Kroll observed the use of HijackLoader malware in attacks to install
infostealer payloads. Lab52 detailed a phishing campaign led by APT-C-36's targeting Colombia where AsyncRAT malware was
deployed. For their part, Kroll documented a "drive-by download" attack campaign through a Bollywood pirated film download
website.
During these two campaigns, the attackers used a ZIP archive containing multiple files, including a legitimate executable signed by
IObit RttHlp.exe, Borland Package Library (BPL) files and several other malicious files. This deployment of HijackLoader also makes
use of new obfuscations techniques to hide the malicious code and to prevent detection by security solutions based on signature
databases.
3.1. HijackLoader
HijackLoader (aka IDAT Loader, DOILoader) is a malicious loader malware observed for the first time in July 2023 by Zscaler
ThreatLabz. This malware uses system calls to evade detection by security solutions, detects several specific processes based on
a blocklist and delays code execution at different steps of its deployment. It also includes multiple embedded modules to simplify
malicious code injection and execution.
HijackLoader is used as a vector to deploy infostealer payloads such as Lumma, Redline, Amadey, Vidar, Raccoon, StealC but also
remote access tools such as AsyncRAT or Remcos.
3.1.1. Attack chain
The modus operandi detailed below is based on the drive-by download campaign observed by Kroll. The attacker lures its victims
through a pirated film download website. When a user tries to download a video, he is redirected to a web page hosted on the
content delivery network (CDN) Bunny that provides a short link bit[.]ly so that he downloads a ZIP file
"""

# ── Change output_type to switch between outputs ──────────────────────────────
# OutputType.SIGMA      → Sigma detection rule (YAML)
# OutputType.YARA       → YARA rule
# OutputType.CTI_REPORT → CTI report + mitigation steps

OUTPUT_TYPE = OutputType.SIGMA   # ← change this line
TOP_K       = 5
SAVE        = True

# ── Run ───────────────────────────────────────────────────────────────────────
def cti_to_rule(
    cti_report: str,
    output_type: OutputType = OutputType.SIGMA,
    top_k: int = 5,
    temperature: float = 0.1,
    save: bool = True,
) -> str:
    label = output_type.value.upper().replace("_", " ")
    print(f"\n{'='*60}")
    print(f"  Output type: {label}")
    print(f"{'='*60}\n")

    print("[1/4] Retrieving relevant context from ChromaDB...")
    context = retrieve_context(cti_report, top_k=top_k)

    print("[2/4] Building prompt...")
    prompt = build_prompt(cti_report, context, output_type=output_type)

    print("[3/4] Generating output with Ollama (sylink:8b)...")
    raw_output = generate_sigma_rule(prompt, temperature=temperature)

    print("[4/4] Cleaning and validating output...")
    cleaned = clean_output(raw_output)
    validate_output(cleaned, output_type)

    if save:
        save_output(cleaned, output_type)

    print(f"\n{'='*60}")
    print(cleaned)
    print(f"{'='*60}")
    return cleaned


result = cti_to_rule(example_cti, output_type=OUTPUT_TYPE, top_k=TOP_K, save=SAVE)


  Output type: SIGMA

[1/4] Retrieving relevant context from ChromaDB...
[2/4] Building prompt...
[3/4] Generating output with Ollama (sylink:8b)...
Sending request to Ollama (sylink/sylink:8b)...
[4/4] Cleaning and validating output...
✅ Sigma: All required fields present.
✅ Saved to: sigma_rules/sigma_20260505_190051.yml

title: Detection of HijackLoader with IObit Driver Booster
id: 3b05d96b-8c8f-4c8d-a6d2-2f3e1f7e5e5c
status: experimental
description: Detects the use of HijackLoader with IObit Driver Booster in ZIP archives containing malicious files.
references:
    - https://www.zscaler.com/resource-center/threat-intelligence/hijackloader-malware
tags:
    - attack.execution
    - attack.t1204
author: SYLink AI
date: 2024-04-05
logsource:
    category: process_creation
    product: windows
detection:
    selection:
        CommandLine|contains:
            - "IObit Driver Booster.exe"
            - "HijackLoader.exe"
        CommandLine|contains:
            - "zip"
            